
# Modelado del Metro CDMX como grafo: DFS y BFS

## Objetivo

Modelar la red del Metro de la Ciudad de México como un **grafo no dirigido** y aplicar:

- **DFS (Depth First Search / búsqueda en profundidad)**.
- **BFS (Breadth First Search / búsqueda en anchura)**.

Rutas solicitadas:

1. **Cuatro Caminos → Pantitlán**
2. **Politécnico → Tasqueña**  
3. **Zapata → Oceanía**

> Los nombres `Taxqueña` y `Ocenia`. En los datasets las estaciones están registradas como **Tasqueña** y **Oceanía**, por lo que se normalizan esos nombres para que sean similares.

Para que DFS y BFS sean comparables se usa **costo unitario**: desplazarse de una estación a la siguiente cuesta 1.  

El dataset de tiempos como análisis secundario para mostrar que **menos estaciones no siempre significa menor tiempo**, debido a los transbordos.



## 1. Carga y análisis de los datasets

Los cinco archivos tienen funciones distintas:

- `metro_cdmx_estaciones.csv`: estructura de líneas, secuencia de estaciones y tiempo medio por tramo. **Es la base para construir el grafo**.
- `metro_cdmx_transbordos.csv`: identifica estaciones de correspondencia. Sirve para validar los transbordos.
- `metro_cdmx_tiempos_estimados.csv`: contiene rutas origen-destino ya modeladas y tiempos estimados. Se usa como referencia de tiempo.
- `metro_cdmx_resumen_lineas.csv`: resume tiempos entre líneas completas; no tiene el detalle necesario para crear aristas estación-estación.
- `metro_cdmx_matriz_mediana_lineas.csv`: matriz agregada de medianas entre líneas; es útil como resumen, pero no para DFS/BFS a nivel estación.


In [1]:

import pandas as pd
from pathlib import Path
from collections import deque, defaultdict

BASE = Path(".")

archivos = {
    "tiempos": BASE / "metro_cdmx_tiempos_estimados.csv",
    "resumen": BASE / "metro_cdmx_resumen_lineas.csv",
    "matriz": BASE / "metro_cdmx_matriz_mediana_lineas.csv",
    "estaciones": BASE / "metro_cdmx_estaciones.csv",
    "transbordos": BASE / "metro_cdmx_transbordos.csv",
}

# Si se ejecuta dentro del entorno donde fueron cargados los archivos:
for k, p in list(archivos.items()):
    if not p.exists():
        alterno = Path("/mnt/data") / p.name
        if alterno.exists():
            archivos[k] = alterno

dfs = {nombre: pd.read_csv(ruta) for nombre, ruta in archivos.items()}

for nombre, df in dfs.items():
    print(f"{nombre:12s} -> {df.shape[0]:5d} filas x {df.shape[1]:2d} columnas")
    print("Columnas:", list(df.columns))
    print()


tiempos      -> 37830 filas x 16 columnas
Columnas: ['id', 'linea_origen', 'estacion_origen', 'linea_destino', 'estacion_destino', 'ruta_lineas', 'estaciones_transbordo', 'num_transbordos', 'num_tramos_estacion', 'tiempo_en_tren_min', 'tiempo_caminata_transbordo_min', 'tiempo_espera_transbordo_min', 'espera_inicial_min', 'tiempo_total_estimado_min', 'ruta_estaciones', 'tipo_estimacion']

resumen      ->   144 filas x  8 columnas
Columnas: ['linea_origen', 'linea_destino', 'viajes_considerados', 'tiempo_min_min', 'tiempo_promedio_min', 'tiempo_mediana_min', 'tiempo_max_min', 'transbordos_promedio']

matriz       ->    12 filas x 13 columnas
Columnas: ['linea_origen', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', '12']

estaciones   ->   195 filas x  6 columnas
Columnas: ['linea', 'secuencia', 'estacion', 'es_correspondencia', 'lineas_conectadas', 'tiempo_medio_tramo_linea_min']

transbordos  ->    28 filas x  3 columnas
Columnas: ['estacion', 'lineas', 'numero_lineas']



In [10]:

estaciones = dfs["estaciones"]
transbordos = dfs["transbordos"]
tiempos = dfs["tiempos"]

print("Registros estación-línea:", len(estaciones))
print("Estaciones únicas:", estaciones["estacion"].nunique())
print("Líneas:", estaciones["linea"].nunique())
print("Estaciones de transbordo registradas:", len(transbordos))

print("\nExtremos de cada línea:")
for linea, grupo in estaciones.groupby("linea", sort=False):
    g = grupo.sort_values("secuencia")
    print(f"Línea {linea:>2}: {g.iloc[0]['estacion']} -> {g.iloc[-1]['estacion']} ({len(g)} estaciones)")


Registros estación-línea: 195
Estaciones únicas: 163
Líneas: 12
Estaciones de transbordo registradas: 28

Extremos de cada línea:
Línea  1: Observatorio -> Pantitlán (20 estaciones)
Línea  2: Cuatro Caminos -> Tasqueña (24 estaciones)
Línea  3: Indios Verdes -> Universidad (21 estaciones)
Línea  4: Martín Carrera -> Santa Anita (10 estaciones)
Línea  5: Pantitlán -> Politécnico (13 estaciones)
Línea  6: El Rosario -> Martín Carrera (11 estaciones)
Línea  7: El Rosario -> Barranca del Muerto (14 estaciones)
Línea  8: Garibaldi/Lagunilla -> Constitución de 1917 (19 estaciones)
Línea  9: Tacubaya -> Pantitlán (12 estaciones)
Línea  A: Pantitlán -> La Paz (10 estaciones)
Línea  B: Ciudad Azteca -> Buenavista (21 estaciones)
Línea 12: Mixcoac -> Tláhuac (20 estaciones)



## 2. Construcción del grafo

Se toma cada línea por separado y se ordena por la columna `secuencia`.

Si dos estaciones son consecutivas en una línea, se agrega una arista en ambos sentidos:

\[
u \leftrightarrow v
\]

El costo de cada arista es:

\[
c(u,v)=1
\]

Las estaciones de correspondencia aparecen repetidas en distintas líneas pero con el mismo nombre. Al usar el **nombre de la estación como nodo**, las líneas quedan conectadas automáticamente en ese punto.


In [18]:

grafo = {}
linea_por_arista = defaultdict(set)

for linea, grupo in estaciones.groupby("linea", sort=False):
    grupo = grupo.sort_values("secuencia")
    lista = grupo["estacion"].tolist()

    for estacion in lista:
        grafo.setdefault(estacion, {})

    for origen, destino in zip(lista, lista[1:]):
        grafo[origen][destino] = 1
        grafo[destino][origen] = 1
        linea_por_arista[frozenset((origen, destino))].add(str(linea))

num_aristas = sum(len(vecinos) for vecinos in grafo.values()) // 2

print("Nodos del grafo:", len(grafo))
print("Aristas no dirigidas:", num_aristas)

# Verificación de conectividad
inicio = next(iter(grafo))
visitados = set()
pila = [inicio]

while pila:
    actual = pila.pop()
    if actual in visitados:
        continue
    visitados.add(actual)
    pila.extend(v for v in grafo[actual] if v not in visitados)

print("Nodos alcanzables desde", inicio + ":", len(visitados))
print("¿La red quedó conectada?:", len(visitados) == len(grafo))


Nodos del grafo: 163
Aristas no dirigidas: 183
Nodos alcanzables desde Observatorio: 163
¿La red quedó conectada?: True



## 3. Clases del problema

Se conserva la misma idea que lo visto en clase: `Problem`, `GraphProblem` y `Node`.

La diferencia principal es que ahora el grafo ya no es el ejemplo de Rumania, sino el grafo generado directamente desde los datos del Metro.


In [25]:

class Problem:
    def __init__(self, initial, goal):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, action):
        raise NotImplementedError

    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        return 1

    def h(self, state):
        return 0


class GraphProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph

    def actions(self, state):
        return list(self.graph[state].keys())

    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]


class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        ruta = []
        node = self
        while node:
            ruta.append(node.state)
            node = node.parent
        return ruta[::-1]

    def expand(self, problem):
        return [self.child_node(problem, action)
                for action in problem.actions(self.state)]

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(
            next_state,
            self,
            action,
            self.path_cost + step_cost
        )



## 4. DFS y BFS

### Observación 

En el visto en clase se revisa si un hijo ya está en `explored`, pero no si **ya fue agregado a la frontera**.  
Eso puede insertar varias veces la misma estación antes de que sea expandida.

No siempre cambia si se encuentra o no una solución, pero sí puede:

- hacer trabajo repetido;
- consumir más memoria;
- volver más difícil interpretar la ruta producida por DFS.

Por eso se usa un conjunto `descubiertos`, que marca una estación **desde el momento en que entra a la frontera**.

### DFS

Usa una pila (`list` con `pop()`), por lo tanto profundiza por una rama antes de regresar.

### BFS

Usa una cola (`deque` con `popleft()`), por lo tanto visita primero todas las estaciones a distancia 1, luego distancia 2, etc.  
Con costo unitario, BFS encuentra un camino con el **mínimo número de tramos**.


In [31]:

def depth_first_graph_search(problem):
    start_node = Node(problem.initial)
    frontier = [start_node]
    descubiertos = {start_node.state}

    while frontier:
        node = frontier.pop()

        if problem.is_goal(node.state):
            return node

        for child in node.expand(problem):
            if child.state not in descubiertos:
                descubiertos.add(child.state)
                frontier.append(child)

    return None


def breadth_first_graph_search(problem):
    start_node = Node(problem.initial)
    frontier = deque([start_node])
    descubiertos = {start_node.state}

    while frontier:
        node = frontier.popleft()

        if problem.is_goal(node.state):
            return node

        for child in node.expand(problem):
            if child.state not in descubiertos:
                descubiertos.add(child.state)
                frontier.append(child)

    return None



## 5. Normalización de nombres y métricas

Se agregan alias para aceptar los nombres tal como aparecen en el enunciado:

- `Taxqueña` → `Tasqueña`
- `Ocenia` → `Oceanía`
- `Pantitlan` → `Pantitlán`
- `Politecnico` → `Politécnico`

También se calculan:

- número de tramos;
- líneas utilizadas;
- estaciones de transbordo;
- tiempo aproximado de tren;
- tiempo total modelado.

El dataset de tiempos usa **3 min de espera inicial** y, por cada transbordo, **3 min de caminata + 3 min de espera**, es decir, 6 min por transbordo.


In [36]:

ALIASES = {
    "Taxqueña": "Tasqueña",
    "Tasqueña": "Tasqueña",
    "Ocenia": "Oceanía",
    "Oceania": "Oceanía",
    "Oceanía": "Oceanía",
    "Pantitlan": "Pantitlán",
    "Pantitlán": "Pantitlán",
    "Politecnico": "Politécnico",
    "Politécnico": "Politécnico",
}

tiempo_tramo_linea = (
    estaciones.groupby("linea")["tiempo_medio_tramo_linea_min"]
    .first()
    .astype(float)
    .to_dict()
)

def normalizar_estacion(nombre):
    return ALIASES.get(nombre, nombre)

def analizar_ruta(ruta):
    lineas_por_tramo = []
    tiempo_tren = 0.0

    for a, b in zip(ruta, ruta[1:]):
        lineas = linea_por_arista[frozenset((a, b))]
        linea = next(iter(lineas))
        lineas_por_tramo.append(linea)
        tiempo_tren += tiempo_tramo_linea[linea]

    lineas_usadas = []
    estaciones_transbordo = []

    if lineas_por_tramo:
        lineas_usadas.append(lineas_por_tramo[0])

        for i in range(1, len(lineas_por_tramo)):
            if lineas_por_tramo[i] != lineas_por_tramo[i - 1]:
                # ruta[i] es la estación entre el tramo anterior y el nuevo
                estaciones_transbordo.append(ruta[i])
                lineas_usadas.append(lineas_por_tramo[i])

    n_transbordos = len(estaciones_transbordo)
    tiempo_total = tiempo_tren + 3 + 6 * n_transbordos

    return {
        "tramos": len(ruta) - 1,
        "lineas": " > ".join(lineas_usadas),
        "transbordos": " > ".join(estaciones_transbordo) if estaciones_transbordo else "Ninguno",
        "num_transbordos": n_transbordos,
        "tiempo_tren_min": round(tiempo_tren, 2),
        "tiempo_total_estimado_min": round(tiempo_total, 2),
    }

def resolver(origen, destino, metodo):
    origen = normalizar_estacion(origen)
    destino = normalizar_estacion(destino)

    if origen not in grafo:
        raise ValueError(f"Estación de origen no encontrada: {origen}")
    if destino not in grafo:
        raise ValueError(f"Estación de destino no encontrada: {destino}")

    problema = GraphProblem(origen, destino, grafo)

    if metodo.upper() == "DFS":
        nodo = depth_first_graph_search(problema)
    elif metodo.upper() == "BFS":
        nodo = breadth_first_graph_search(problema)
    else:
        raise ValueError("Método debe ser DFS o BFS")

    ruta = nodo.path()
    datos = analizar_ruta(ruta)

    return {
        "origen": origen,
        "destino": destino,
        "metodo": metodo.upper(),
        "ruta": ruta,
        **datos
    }



# 6. Ejecución de lo solicitado en la tarea


In [40]:

casos = [
    ("Cuatro Caminos", "Pantitlán"),
    ("Politécnico", "Taxqueña"),
    ("Zapata", "Ocenia"),
]

resultados = []

for origen, destino in casos:
    print("=" * 100)
    print(f"{origen} -> {destino}")

    for metodo in ("DFS", "BFS"):
        r = resolver(origen, destino, metodo)
        resultados.append(r)

        print(f"\n{metodo}")
        print("Ruta:")
        print(" -> ".join(r["ruta"]))
        print("Tramos:", r["tramos"])
        print("Líneas:", r["lineas"])
        print("Transbordos:", r["transbordos"])
        print("Número de transbordos:", r["num_transbordos"])
        print("Tiempo aproximado en tren:", r["tiempo_tren_min"], "min")
        print("Tiempo total estimado:", r["tiempo_total_estimado_min"], "min")
    print()


Cuatro Caminos -> Pantitlán

DFS
Ruta:
Cuatro Caminos -> Panteones -> Tacuba -> San Joaquín -> Polanco -> Auditorio -> Constituyentes -> Tacubaya -> Patriotismo -> Chilpancingo -> Centro Médico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Mixiuhca -> Velódromo -> Ciudad Deportiva -> Puebla -> Pantitlán
Tramos: 18
Líneas: 2 > 7 > 9
Transbordos: Tacuba > Tacubaya
Número de transbordos: 2
Tiempo aproximado en tren: 57.71 min
Tiempo total estimado: 72.71 min

BFS
Ruta:
Cuatro Caminos -> Panteones -> Tacuba -> San Joaquín -> Polanco -> Auditorio -> Constituyentes -> Tacubaya -> Patriotismo -> Chilpancingo -> Centro Médico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Mixiuhca -> Velódromo -> Ciudad Deportiva -> Puebla -> Pantitlán
Tramos: 18
Líneas: 2 > 7 > 9
Transbordos: Tacuba > Tacubaya
Número de transbordos: 2
Tiempo aproximado en tren: 57.71 min
Tiempo total estimado: 72.71 min

Politécnico -> Taxqueña

DFS
Ruta:
Politécnico -> Instituto del Petróleo -> Lindavista -> Deportivo 18 de M


## 7. Tabla comparativa DFS vs BFS


In [41]:

tabla = pd.DataFrame([
    {
        "Origen": r["origen"],
        "Destino": r["destino"],
        "Método": r["metodo"],
        "Tramos": r["tramos"],
        "Líneas": r["lineas"],
        "Núm. transbordos": r["num_transbordos"],
        "Transbordos": r["transbordos"],
        "Tiempo estimado (min)": r["tiempo_total_estimado_min"],
    }
    for r in resultados
])

tabla


,Origen,Destino,Método,Tramos,Líneas,Núm. transbordos,Transbordos,Tiempo estimado (min)
0,Cuatro Caminos,Pantitlán,DFS,18,2 > 7 > 9,2,Tacuba > Tacubaya,72.71
1,Cuatro Caminos,Pantitlán,BFS,18,2 > 7 > 9,2,Tacuba > Tacubaya,72.71
2,Politécnico,Tasqueña,DFS,46,5 > 6 > 4 > 5 > 3 > B > 5 > 9 > 7 > 12 > 2,10,Instituto del Petróleo > Martín Carrera > Cons...,190.76
3,Politécnico,Tasqueña,BFS,20,5 > 3 > 2,2,La Raza > Hidalgo,65.62
4,Zapata,Oceanía,DFS,53,12 > 8 > 9 > 7 > 6 > 4 > 5 > 3 > B,8,Atlalilco > Chabacano > Tacubaya > El Rosario ...,202.19
5,Zapata,Oceanía,BFS,13,3 > 9 > 4 > 1 > B,4,Centro Médico > Jamaica > Candelaria > San Lázaro,62.09



## 8. Comparación contra el dataset de tiempos estimados

Esta comparación **no cambia el objetivo de DFS/BFS**. Solo sirve para comprobar una idea importante:

- BFS minimiza estaciones/tramos porque usamos costo 1.
- El menor tiempo puede ser distinto, porque el tiempo depende de la línea y de los transbordos.


In [42]:

referencias = []

for origen, destino in [
    ("Cuatro Caminos", "Pantitlán"),
    ("Politécnico", "Tasqueña"),
    ("Zapata", "Oceanía"),
]:
    candidatos = tiempos[
        (tiempos["estacion_origen"] == origen) &
        (tiempos["estacion_destino"] == destino)
    ].copy()

    mejor = candidatos.sort_values("tiempo_total_estimado_min").iloc[0]

    referencias.append({
        "Origen": origen,
        "Destino": destino,
        "Ruta de líneas más rápida en dataset": mejor["ruta_lineas"],
        "Transbordos": mejor["estaciones_transbordo"],
        "Tramos": int(mejor["num_tramos_estacion"]),
        "Tiempo dataset (min)": float(mejor["tiempo_total_estimado_min"]),
        "Ruta estaciones": mejor["ruta_estaciones"],
    })

pd.DataFrame(referencias)


,Origen,Destino,Ruta de líneas más rápida en dataset,Transbordos,Tramos,Tiempo dataset (min),Ruta estaciones
0,Cuatro Caminos,Pantitlán,2 > 1,Pino Suárez,22,61.40,Cuatro Caminos > Panteones > Tacuba > Cuitláhu...
1,Politécnico,Tasqueña,5 > 3 > 2,La Raza > Hidalgo,20,65.63,Politécnico > Instituto del Petróleo > Autobus...
2,Zapata,Oceanía,3 > B,Guerrero,18,53.40,Zapata > División del Norte > Eugenia > Etiopí...
